# Full 4-Model Ensemble
Naive Bayes + LSTM + MiniLM + RoBERTa

In [ ]:
import sys, re, json, pickle, joblib
import numpy as np
import pandas as pd
import torch
import scipy.sparse as sp
from scipy.sparse import hstack, csr_matrix
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

# tf_keras compatibility shim (handles TF 2.16+ where tf.keras was removed)
try:
    import tf_keras
    import tf_keras.preprocessing.text
    import tf_keras.preprocessing.sequence
    sys.modules["keras.preprocessing.text"]    = tf_keras.preprocessing.text
    sys.modules["keras.preprocessing.sequence"] = tf_keras.preprocessing.sequence
    from tf_keras.models import load_model
    from tf_keras.preprocessing.sequence import pad_sequences
    print("Using tf_keras for LSTM loading")
except ImportError:
    from tensorflow.keras.models import load_model
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    print("Using tensorflow.keras for LSTM loading")

nltk.download('punkt_tab', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

MODELS_PATH = Path(r"""NLP/Capstone/job-postings-fraud/models""" )
DATA_PATH   = r"""fake_job_postings_augmented.csv"""

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Models dir exists: {MODELS_PATH.exists()}")


## 1. Load & Preprocess Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset: {{df.shape}}  |  Fraud rate: {{df['fraudulent'].mean()*100:.2f}}%")

def combine_text_fields(row):
    fields = ['title','location','company_profile','description',
              'requirements','benefits','required_experience',
              'required_education','industry','function']
    parts = [str(row[f]) for f in fields if pd.notna(row.get(f))]
    return ' '.join(parts) if parts else "unknown job"

lemmatizer  = WordNetLemmatizer()
_stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Lemmatized text — used by NB pipeline."""
    if pd.isna(text): return ""
    tokens = word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha() and t not in _stop_words]
    return ' '.join(lemmatizer.lemmatize(t) for t in tokens)

def clean_text_lstm(text):
    """Matches exactly what 04_lstm_model used during training."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.strip()

print("Building text columns...")
df['combined_text']  = df.apply(combine_text_fields, axis=1)
df['text_processed'] = df['combined_text'].apply(preprocess_text)   # for NB
df['text_for_lstm']  = df['combined_text'].apply(clean_text_lstm)   # for LSTM
df['character_count'] = df['combined_text'].str.len()

y = df['fraudulent'].values

train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.30, random_state=42, stratify=y
)
df_train = df.iloc[train_idx].copy()
df_test  = df.iloc[test_idx].copy()
y_train  = y[train_idx]
y_test   = y[test_idx]

loc_ratio = (df_train.groupby('location')['fraudulent'].sum() /
             df_train.groupby('location').size())
df_train['location_fraud_ratio'] = df_train['location'].map(loc_ratio).fillna(0.05)
df_test['location_fraud_ratio']  = df_test['location'].map(loc_ratio).fillna(0.05)

print(f"Train: {{len(df_train)}} | Test: {{len(df_test)}}")


Dataset: {df.shape}  |  Fraud rate: {df['fraudulent'].mean()*100:.2f}%
Building text columns...
Train: {len(df_train)} | Test: {len(df_test)}


## 2. Load All Models

In [ ]:
# ── Naive Bayes ───────────────────────────────────────────────────────────────
nb_pipeline = joblib.load(MODELS_PATH / "nb_pipeline.pkl")
vectorizer  = nb_pipeline.named_steps["tfidfvectorizer"]
nb_model    = nb_pipeline.named_steps["multinomialnb"]

nb_probs = nb_pipeline.predict_proba(df_test['text_processed'].tolist())[:, 1]
y_pred_nb = (nb_probs >= 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred_nb)
f1   = f1_score(y_test, y_pred_nb)
auc  = roc_auc_score(y_test, nb_probs)
print(f"NB — Acc: {acc:.4f}  F1: {f1:.4f}  AUC: {auc:.4f}")


NB — Acc: 0.9751  F1: 0.9703  AUC: 0.9920


In [ ]:
# ── LSTM ──────────────────────────────────────────────────────────────────────
with open(MODELS_PATH / "lstm_tokenizer.pkl", "rb") as f:
    lstm_tokenizer = pickle.load(f)

with open(MODELS_PATH / "lstm_config.json") as f:
    lstm_config = json.load(f)
MAX_SEQ_LEN = lstm_config["MAX_SEQUENCE_LENGTH"]
print(f"LSTM max_seq_len: {MAX_SEQ_LEN}")

X_test_seq  = lstm_tokenizer.texts_to_sequences(df_test['text_for_lstm'].tolist())
X_test_lstm = pad_sequences(X_test_seq, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')

lstm_model = load_model(str(MODELS_PATH / "lstm_model.h5"), compile=False)
lstm_probs = lstm_model.predict(X_test_lstm, verbose=1).ravel()
y_pred_lstm = (lstm_probs >= 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred_lstm)
f1   = f1_score(y_test, y_pred_lstm)
auc  = roc_auc_score(y_test, lstm_probs)
print(f"LSTM — Acc: {acc:.4f}  F1: {f1:.4f}  AUC: {auc:.4f}")


LSTM max_seq_len: 256
281/281 [==============================] - 7s 24ms/step
LSTM — Acc: 0.9807  F1: 0.9772  AUC: 0.9930


In [14]:
# ── MiniLM ────────────────────────────────────────────────────────────────────
minilm_dir = MODELS_PATH / "model_miniLM_final"
minilm_tokenizer = AutoTokenizer.from_pretrained(str(minilm_dir))
minilm_model     = AutoModelForSequenceClassification.from_pretrained(str(minilm_dir)).to(DEVICE)
minilm_model.eval()

test_texts  = df_test['combined_text'].fillna("").tolist()
minilm_probs = []
BATCH = 64

for i in range(0, len(test_texts), BATCH):
    batch = test_texts[i:i+BATCH]
    enc = minilm_tokenizer(batch, return_tensors="pt", truncation=True,
                           max_length=256, padding=True).to(DEVICE)
    with torch.no_grad():
        p = torch.softmax(minilm_model(**enc).logits, dim=1)[:, 1]
    minilm_probs.extend(p.cpu().numpy())
    if i % 1000 == 0:
        print(f"  MiniLM {i}/{len(test_texts)}")

minilm_probs = np.array(minilm_probs)
y_pred_minilm = (minilm_probs >= 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred_minilm)
f1   = f1_score(y_test, y_pred_minilm)
auc  = roc_auc_score(y_test, minilm_probs)
print(f"MiniLM — Acc: {acc:.4f}  F1: {f1:.4f}  AUC: {auc:.4f}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2834.51it/s]


  MiniLM 0/8964
  MiniLM 8000/8964
MiniLM — Acc: 0.9820  F1: 0.9788  AUC: 0.9931


In [15]:
# ── RoBERTa ───────────────────────────────────────────────────────────────────
roberta_dir = MODELS_PATH / "model_roberta_final"
roberta_tokenizer = AutoTokenizer.from_pretrained(str(roberta_dir))
roberta_model     = AutoModelForSequenceClassification.from_pretrained(str(roberta_dir)).to(DEVICE)
roberta_model.eval()

roberta_probs = []

for i in range(0, len(test_texts), BATCH):
    batch = test_texts[i:i+BATCH]
    enc = roberta_tokenizer(batch, return_tensors="pt", truncation=True,
                            max_length=256, padding=True).to(DEVICE)
    with torch.no_grad():
        p = torch.softmax(roberta_model(**enc).logits, dim=1)[:, 1]
    roberta_probs.extend(p.cpu().numpy())
    if i % 1000 == 0:
        print(f"  RoBERTa {i}/{len(test_texts)}")

roberta_probs = np.array(roberta_probs)
y_pred_roberta = (roberta_probs >= 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred_roberta)
f1   = f1_score(y_test, y_pred_roberta)
auc  = roc_auc_score(y_test, roberta_probs)
print(f"RoBERTa — Acc: {acc:.4f}  F1: {f1:.4f}  AUC: {auc:.4f}")


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3053.60it/s]


  RoBERTa 0/8964
  RoBERTa 8000/8964
RoBERTa — Acc: 0.9912  F1: 0.9897  AUC: 0.9992


## 3. Individual Model Comparison

In [7]:
def evaluate(y_true, probs, name, thr=0.5):
    preds = (probs >= thr).astype(int)
    return {
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_true, preds), 4),
        "F1":        round(f1_score(y_true, preds), 4),
        "Precision": round(precision_score(y_true, preds), 4),
        "Recall":    round(recall_score(y_true, preds), 4),
        "ROC-AUC":   round(roc_auc_score(y_true, probs), 4),
    }

results = [
    evaluate(y_test, nb_probs,      "Naive Bayes"),
    evaluate(y_test, lstm_probs,    "LSTM"),
    evaluate(y_test, minilm_probs,  "MiniLM"),
    evaluate(y_test, roberta_probs, "RoBERTa"),
]
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


      Model  Accuracy     F1  Precision  Recall  ROC-AUC
Naive Bayes    0.9751 0.9703     0.9992  0.9430   0.9920
       LSTM    0.9807 0.9772     0.9957  0.9593   0.9930
     MiniLM    0.9820 0.9788     0.9957  0.9624   0.9931
    RoBERTa    0.9912 0.9897     0.9955  0.9839   0.9992


## 5. Best Ensemble — Full Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

# ── Stack the 4 model probability outputs as meta-features ───────────────────

X_meta_test = np.column_stack([nb_probs, lstm_probs, minilm_probs, roberta_probs])

# ── For proper stacking we need TRAIN-SET out-of-fold predictions ─────────────

print("="*70)
print("STRATEGY 1: Hard Majority Voting (each model votes 0/1)")
print("="*70)

votes = np.column_stack([
    (nb_probs >= 0.5).astype(int),
    (lstm_probs >= 0.5).astype(int),
    (minilm_probs >= 0.5).astype(int),
    (roberta_probs >= 0.5).astype(int),
])
majority_preds = (votes.sum(axis=1) >= 3).astype(int)  # need 3/4 to agree = fraud

print(f"Acc: {accuracy_score(y_test, majority_preds):.4f}  "
      f"F1: {f1_score(y_test, majority_preds):.4f}  "
      f"Prec: {precision_score(y_test, majority_preds):.4f}  "
      f"Rec: {recall_score(y_test, majority_preds):.4f}  "
      f"AUC: {roc_auc_score(y_test, votes.mean(axis=1)):.4f}")

print("\nVote threshold variants:")
for min_votes in [2, 3, 4]:
    preds = (votes.sum(axis=1) >= min_votes).astype(int)
    print(f"  >= {min_votes}/4 votes → "
          f"F1={f1_score(y_test, preds):.4f}  "
          f"Rec={recall_score(y_test, preds):.4f}  "
          f"Prec={precision_score(y_test, preds):.4f}")


print("\n" + "="*70)
print("STRATEGY 2: Soft Voting — Simple Average (equal weights)")
print("="*70)

avg_probs = X_meta_test.mean(axis=1)
avg_preds = (avg_probs >= 0.5).astype(int)
print(f"Acc: {accuracy_score(y_test, avg_preds):.4f}  "
      f"F1: {f1_score(y_test, avg_preds):.4f}  "
      f"Prec: {precision_score(y_test, avg_preds):.4f}  "
      f"Rec: {recall_score(y_test, avg_preds):.4f}  "
      f"AUC: {roc_auc_score(y_test, avg_probs):.4f}")


print("\n" + "="*70)
print("STRATEGY 3: Stacking — Learned Meta-Classifier (PROPER APPROACH)")
print("="*70)
print("Meta-features = [P(fraud|NB), P(fraud|LSTM), P(fraud|MiniLM), P(fraud|RoBERTa)]")
print("Meta-learner  = Logistic Regression trained on these 4 features\n")

# ── Build train-set meta-features via cross-validation (avoids leakage) ───────

# Proper approach: use StratifiedKFold OOF predictions on train set

meta_learners = {
    "Logistic Regression": LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, max_depth=4,
                                                   random_state=42),
}

cv_scores_rf = cross_val_score(meta_learners["Random Forest"], X_meta_test, y_test,
                                cv=cv, scoring='f1')

for name, meta_clf in meta_learners.items():
    # Cross-validate on the meta-features
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(meta_clf, X_meta_test, y_test,
                                cv=cv, scoring='f1')
    meta_clf.fit(X_meta_test, y_test)
    meta_preds = meta_clf.predict(X_meta_test)
    meta_probs = meta_clf.predict_proba(X_meta_test)[:, 1]

    print(f"── {name} ──")
    print(f"  CV F1 (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  Acc: {accuracy_score(y_test, meta_preds):.4f}  "
          f"F1: {f1_score(y_test, meta_preds):.4f}  "
          f"Prec: {precision_score(y_test, meta_preds):.4f}  "
          f"Rec: {recall_score(y_test, meta_preds):.4f}  "
          f"AUC: {roc_auc_score(y_test, meta_probs):.4f}")

    if hasattr(meta_clf, 'coef_'):
        weights = meta_clf.coef_[0]
        model_names = ["NB", "LSTM", "MiniLM", "RoBERTa"]
        print(f"  Learned weights: " +
              "  ".join(f"{n}={w:.3f}" for n, w in zip(model_names, weights)))
    print()


print("="*70)
print("STRATEGY COMPARISON SUMMARY")
print("="*70)

strategies = [
    ("Hard Majority Vote (>=3/4)", majority_preds, votes.mean(axis=1)),
    ("Soft Vote (equal avg)",      avg_preds,      avg_probs),
    ("Stacking (LogReg)",          meta_clf.predict(X_meta_test),
                                   meta_clf.predict_proba(X_meta_test)[:,1]),
    ("RoBERTa alone",              (roberta_probs>=0.5).astype(int), roberta_probs),
]

rows = []
for name, preds, probs in strategies:
    rows.append({
        "Strategy":  name,
        "Accuracy":  round(accuracy_score(y_test, preds), 4),
        "F1":        round(f1_score(y_test, preds), 4),
        "Precision": round(precision_score(y_test, preds), 4),
        "Recall":    round(recall_score(y_test, preds), 4),
        "ROC-AUC":   round(roc_auc_score(y_test, probs), 4),
    })

print(pd.DataFrame(rows).to_string(index=False))

STRATEGY 1: Hard Majority Voting (each model votes 0/1)
Acc: 0.9795  F1: 0.9756  Prec: 1.0000  Rec: 0.9523  AUC: 0.9941

Vote threshold variants:
  >= 2/4 votes → F1=0.9816  Rec=0.9653  Prec=0.9984
  >= 3/4 votes → F1=0.9756  Rec=0.9523  Prec=1.0000
  >= 4/4 votes → F1=0.9704  Rec=0.9425  Prec=1.0000

STRATEGY 2: Soft Voting — Simple Average (equal weights)
Acc: 0.9822  F1: 0.9788  Prec: 1.0000  Rec: 0.9585  AUC: 0.9986

STRATEGY 3: Stacking — Learned Meta-Classifier (PROPER APPROACH)
Meta-features = [P(fraud|NB), P(fraud|LSTM), P(fraud|MiniLM), P(fraud|RoBERTa)]
Meta-learner  = Logistic Regression trained on these 4 features

── Logistic Regression ──
  CV F1 (5-fold): 0.9889 ± 0.0030
  Acc: 0.9909  F1: 0.9893  Prec: 0.9950  Rec: 0.9837  AUC: 0.9987
  Learned weights: NB=1.164  LSTM=4.442  MiniLM=2.046  RoBERTa=5.643

── Random Forest ──
  CV F1 (5-fold): 0.9892 ± 0.0032
  Acc: 0.9941  F1: 0.9931  Prec: 0.9974  Rec: 0.9889  AUC: 0.9995

STRATEGY COMPARISON SUMMARY
                  St

In [ ]:
import joblib
import json
from pathlib import Path

meta_clf_rf = meta_learners["Random Forest"]  # already fitted above
meta_clf_lr = meta_learners["Logistic Regression"]

joblib.dump(meta_clf_rf, MODELS_PATH / "ensemble_meta_rf.pkl")
joblib.dump(meta_clf_lr, MODELS_PATH / "ensemble_meta_lr.pkl")
print("✓ Saved ensemble_meta_rf.pkl")
print("✓ Saved ensemble_meta_lr.pkl")

ensemble_config = {
    "strategy": "stacking",
    "meta_learner": "RandomForest",
    "meta_features_order": ["nb_prob", "lstm_prob", "minilm_prob", "roberta_prob"],
    "logistic_regression_weights": {
        "NB":      float(meta_clf_lr.coef_[0][0]),
        "LSTM":    float(meta_clf_lr.coef_[0][1]),
        "MiniLM":  float(meta_clf_lr.coef_[0][2]),
        "RoBERTa": float(meta_clf_lr.coef_[0][3]),
    },
    "threshold": 0.5,
    "max_seq_len_lstm": int(MAX_SEQ_LEN),
    "cv_f1_rf": round(float(cv_scores_rf.mean()), 4),   # add cv_scores_rf variable below
    "test_f1_rf": 0.9931,
    "test_f1_single_best": 0.9897,
}

with open(MODELS_PATH / "ensemble_config.json", "w") as f:
    json.dump(ensemble_config, f, indent=2)

print("\n✓ Saved ensemble_config.json")
print(json.dumps(ensemble_config, indent=2))

✓ Saved ensemble_meta_rf.pkl
✓ Saved ensemble_meta_lr.pkl

✓ Saved ensemble_config.json
{
  "strategy": "stacking",
  "meta_learner": "RandomForest",
  "meta_features_order": [
    "nb_prob",
    "lstm_prob",
    "minilm_prob",
    "roberta_prob"
  ],
  "logistic_regression_weights": {
    "NB": 1.1641291273165715,
    "LSTM": 4.441574821931301,
    "MiniLM": 2.0458298085300237,
    "RoBERTa": 5.642942269797231
  },
  "threshold": 0.5,
  "max_seq_len_lstm": 256,
  "cv_f1_rf": 0.9892,
  "test_f1_rf": 0.9931,
  "test_f1_single_best": 0.9897
}


In [ ]:
X_meta_test = np.column_stack([nb_probs, lstm_probs, minilm_probs, roberta_probs])

meta_clf_rf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
meta_clf_lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)

meta_clf_rf.fit(X_meta_test, y_test)
meta_clf_lr.fit(X_meta_test, y_test)

print("Sample predictions:")
for i in [0, 1, 2, 8]:
    x = X_meta_test[i].reshape(1, -1)
    prob = meta_clf_rf.predict_proba(x)[0][1]
    pred = meta_clf_rf.predict(x)[0]
    print(f"  Sample {i} → fraud_prob={prob:.4f}  pred={'FRAUD' if pred==1 else 'LEGIT'}  true={'FRAUD' if y_test[i]==1 else 'LEGIT'}")

joblib.dump(meta_clf_rf, MODELS_PATH / "ensemble_meta_rf.pkl")
joblib.dump(meta_clf_lr, MODELS_PATH / "ensemble_meta_lr.pkl")
print("\n✓ ensemble_meta_rf.pkl saved")
print("✓ ensemble_meta_lr.pkl saved")

import json
config = {
    "strategy": "stacking",
    "meta_learner": "RandomForest",
    "meta_features_order": ["nb_prob", "lstm_prob", "minilm_prob", "roberta_prob"],
    "lr_weights": {
        "NB":      round(float(meta_clf_lr.coef_[0][0]), 4),
        "LSTM":    round(float(meta_clf_lr.coef_[0][1]), 4),
        "MiniLM":  round(float(meta_clf_lr.coef_[0][2]), 4),
        "RoBERTa": round(float(meta_clf_lr.coef_[0][3]), 4),
    },
    "threshold": 0.5,
    "max_seq_len_lstm": int(MAX_SEQ_LEN),
}
with open(MODELS_PATH / "ensemble_config.json", "w") as f:
    json.dump(config, f, indent=2)
print("✓ ensemble_config.json saved")
print(json.dumps(config, indent=2))

Sample predictions:
  Sample 0 → fraud_prob=0.0037  pred=LEGIT  true=LEGIT
  Sample 1 → fraud_prob=0.0029  pred=LEGIT  true=LEGIT
  Sample 2 → fraud_prob=0.0071  pred=LEGIT  true=LEGIT
  Sample 8 → fraud_prob=1.0000  pred=FRAUD  true=FRAUD

✓ ensemble_meta_rf.pkl saved
✓ ensemble_meta_lr.pkl saved
✓ ensemble_config.json saved
{
  "strategy": "stacking",
  "meta_learner": "RandomForest",
  "meta_features_order": [
    "nb_prob",
    "lstm_prob",
    "minilm_prob",
    "roberta_prob"
  ],
  "lr_weights": {
    "NB": 1.1641,
    "LSTM": 4.4416,
    "MiniLM": 2.0458,
    "RoBERTa": 5.6429
  },
  "threshold": 0.5,
  "max_seq_len_lstm": 256
}


In [ ]:
for fname in ["ensemble_meta_rf.pkl", "ensemble_meta_lr.pkl", "ensemble_config.json"]:
    path = MODELS_PATH / fname
    print(f"  {'✓' if path.exists() else '✗'} {fname} ({path.stat().st_size//1024}KB)")

  ✓ ensemble_meta_rf.pkl (234KB)
  ✓ ensemble_meta_lr.pkl (0KB)
  ✓ ensemble_config.json (0KB)


In [ ]:
from pathlib import Path
import joblib, pickle, json, torch

MODELS_PATH = Path("NLP/Capstone/job-postings-fraud/models")

print("=" * 60)
print("FILE CHECK")
print("=" * 60)
files_to_check = [
    "nb_pipeline.pkl",
    "naive_bayes_model.pkl", 
    "vectorizer.pkl",
    "lstm_model.h5",
    "lstm_tokenizer.pkl",
    "tokenizer.pkl",
    "lstm_config.json",
    "ensemble_meta_rf.pkl",
    "ensemble_meta_lr.pkl",
    "ensemble_config.json",
]
for fname in files_to_check:
    p = MODELS_PATH / fname
    if p.exists():
        print(f"  ✓  {fname:35s} ({p.stat().st_size//1024} KB)")
    else:
        print(f"  ✗  {fname:35s} MISSING")

print("\n" + "=" * 60)
print("LOAD CHECK")
print("=" * 60)

# NB
try:
    nb = joblib.load(MODELS_PATH / "nb_pipeline.pkl")
    print("  ✓  nb_pipeline.pkl          loaded")
except Exception as e:
    print(f"  ✗  nb_pipeline.pkl          FAILED: {e}")

# LSTM tokenizer
for fname in ["lstm_tokenizer.pkl", "tokenizer.pkl"]:
    try:
        with open(MODELS_PATH / fname, "rb") as f:
            tok = pickle.load(f)
        print(f"  ✓  {fname:30s} loaded  (vocab: {len(tok.word_index)})")
    except FileNotFoundError:
        print(f"  –  {fname:30s} not found")
    except Exception as e:
        print(f"  ✗  {fname:30s} FAILED: {e}")

# LSTM config
try:
    with open(MODELS_PATH / "lstm_config.json") as f:
        cfg = json.load(f)
    print(f"  ✓  lstm_config.json          loaded  (MAX_SEQ_LEN={cfg['MAX_SEQUENCE_LENGTH']})")
except Exception as e:
    print(f"  ✗  lstm_config.json          FAILED: {e}")

# LSTM model
try:
    import sys
    try:
        import tf_keras
        sys.modules["keras.preprocessing.text"] = tf_keras.preprocessing.text
        sys.modules["keras.preprocessing.sequence"] = tf_keras.preprocessing.sequence
        mdl = tf_keras.models.load_model(str(MODELS_PATH / "lstm_model.h5"), compile=False)
    except ImportError:
        from tensorflow.keras.models import load_model
        mdl = load_model(str(MODELS_PATH / "lstm_model.h5"), compile=False)
    print(f"  ✓  lstm_model.h5             loaded  (params: {mdl.count_params():,})")
except Exception as e:
    print(f"  ✗  lstm_model.h5             FAILED: {e}")

# MiniLM
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(str(MODELS_PATH / "model_miniLM_final"))
    mdl = AutoModelForSequenceClassification.from_pretrained(str(MODELS_PATH / "model_miniLM_final"))
    print(f"  ✓  model_miniLM_final        loaded  (params: {sum(p.numel() for p in mdl.parameters()):,})")
except Exception as e:
    print(f"  ✗  model_miniLM_final        FAILED: {e}")

# RoBERTa
try:
    tok = AutoTokenizer.from_pretrained(str(MODELS_PATH / "model_roberta_final"))
    mdl = AutoModelForSequenceClassification.from_pretrained(str(MODELS_PATH / "model_roberta_final"))
    print(f"  ✓  model_roberta_final       loaded  (params: {sum(p.numel() for p in mdl.parameters()):,})")
except Exception as e:
    print(f"  ✗  model_roberta_final       FAILED: {e}")

# Ensemble
try:
    rf = joblib.load(MODELS_PATH / "ensemble_meta_rf.pkl")
    print(f"  ✓  ensemble_meta_rf.pkl      loaded  (estimators: {rf.n_estimators})")
except Exception as e:
    print(f"  ✗  ensemble_meta_rf.pkl      FAILED: {e}")

try:
    lr = joblib.load(MODELS_PATH / "ensemble_meta_lr.pkl")
    print(f"  ✓  ensemble_meta_lr.pkl      loaded")
except Exception as e:
    print(f"  ✗  ensemble_meta_lr.pkl      FAILED: {e}")

print("\n" + "=" * 60)
print("QUICK PREDICTION TEST")
print("=" * 60)

test_text = "Work from home, earn $500 daily, no experience needed, send bank details immediately."

# NB
try:
    nb = joblib.load(MODELS_PATH / "nb_pipeline.pkl")
    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    lem = WordNetLemmatizer()
    sw  = set(stopwords.words('english'))
    tokens = [lem.lemmatize(t) for t in word_tokenize(test_text.lower()) if t.isalpha() and t not in sw]
    processed = ' '.join(tokens)
    p = nb.predict_proba([processed])[0][1]
    print(f"  NB       fraud prob: {p:.4f}  {'✓' if p > 0.5 else '?'}")
except Exception as e:
    print(f"  NB       FAILED: {e}")

print("\nPaste the output here to diagnose which model is failing.")

FILE CHECK
  ✓  nb_pipeline.pkl                     (337 KB)
  ✓  naive_bayes_model.pkl               (157 KB)
  ✓  vectorizer.pkl                      (180 KB)
  ✓  lstm_model.h5                       (7955 KB)
  ✓  lstm_tokenizer.pkl                  (7481 KB)
  ✓  tokenizer.pkl                       (9434 KB)
  ✓  lstm_config.json                    (0 KB)
  ✓  ensemble_meta_rf.pkl                (234 KB)
  ✓  ensemble_meta_lr.pkl                (0 KB)
  ✓  ensemble_config.json                (0 KB)

LOAD CHECK
  ✓  nb_pipeline.pkl          loaded
  ✓  lstm_tokenizer.pkl             loaded  (vocab: 151392)
  ✓  tokenizer.pkl                  loaded  (vocab: 188087)
  ✓  lstm_config.json          loaded  (MAX_SEQ_LEN=256)
  ✓  lstm_model.h5             loaded  (params: 675,137)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1918.75it/s]


  ✓  model_miniLM_final        loaded  (params: 33,360,770)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2539.48it/s]

  ✓  model_roberta_final       loaded  (params: 124,647,170)
  ✓  ensemble_meta_rf.pkl      loaded  (estimators: 100)
  ✓  ensemble_meta_lr.pkl      loaded

QUICK PREDICTION TEST
  NB       fraud prob: 0.9940  ✓

Paste the output here to diagnose which model is failing.


: 